In [1]:
pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 27.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 104.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 16.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━

In [2]:
!nvidia-smi
!python --version

Mon Oct 12 12:07:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q vllm httpx
import vllm
print("vLLM version:", vllm.__version__)

vLLM version: 0.31.0


In [5]:
import os, sys, time, json, shutil, asyncio, statistics, subprocess
import httpx

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
PORT = 8000
MAX_NEW = 48
BASE_ARGS = ["--dtype", "half",                    # T4 has no bfloat16 support -> use float16
             "--max-model-len", "2048",            # max prompt + output tokens per request
             "--gpu-memory-utilization", "0.85"]   # share of GPU memory vLLM may use (weights + KV cache)

QUESTIONS = [
    "Answer with one word: what colour is the sky on a clear day?",
    "What is 12 times 7? Reply with only the number.",
    "Write three sentences about why the ocean is salty.",
    "List five fruits, comma separated.",
    "Explain in detail how a refrigerator keeps food cold.",
    "Say hello.",
    "Give a short definition of photosynthesis.",
    "Write a short paragraph about the history of the bicycle.",
    "What is the capital of Japan? One word.",
    "Describe how rain forms, step by step.",
    "Translate 'good morning' into French.",
    "Write a haiku about coffee.",
    "Explain what a database index is and why it speeds up queries.",
    "Name the planet closest to the sun.",
    "Give three tips for writing clean Python code.",
    "Summarise the water cycle in two sentences.",
]
print("vllm command found at:", shutil.which("vllm"))

vllm command found at: /usr/local/bin/vllm


In [6]:
def start_vllm(extra_args=(), port=PORT, timeout_s=900):
    log_path = f"vllm_{port}.log"
    log = open(log_path, "w")
    cmd = ["vllm", "serve", MODEL_ID, "--port", str(port), *BASE_ARGS, *extra_args]
    print("starting:", " ".join(cmd))
    
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    url = f"http://127.0.0.1:{port}"
    
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < timeout_s:
        if proc.poll() is not None:
            log.close()
            raise RuntimeError("vLLM exited:\n" + open(log_path).read()[-4000:])
        
        try:
            if httpx.get(url + "/health", timeout=2).status_code == 200:
                print(f"vLLM ready after {time.perf_counter() - t0:.0f}s")
                for line in open(log_path):
                    if "KV cache size" in line or "Maximum concurrency" in line:
                        print("  log:", line.strip()[-160:])
                return proc, url
        except httpx.HTTPError:
            pass
        time.sleep(2)
    proc.terminate()
    raise TimeoutError("vLLM did not start in time; see " + log_path)

def stop_vllm(proc):
    proc.terminate()
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        proc.kill()
    print("vLLM stopped")
    
proc, url = start_vllm()
print("models:", httpx.get(url + "/v1/models").json()["data"][0]["id"])
    

starting: vllm serve Qwen/Qwen2.5-0.5B-Instruct --port 8000 --dtype half --max-model-len 2048 --gpu-memory-utilization 0.85
vLLM ready after 162s
  log: To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.8692. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
  log: (EngineCore pid=7912) INFO 10-12 12:31:51 [kv_cache_utils.py:2464] GPU KV cache size: 929,184 tokens, Maximum concurrency for 2,048 tokens per request: 453.70x
models: Qwen/Qwen2.5-0.5B-Instruct


In [22]:
async def stream_chat(client, url, question, max_tokens=MAX_NEW):
    """
        Send one chat request with streaming ON, and time the first token and the whole answer.
    """
    body = {
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": question}],
        "max_tokens": max_tokens,
        "temperature": 0,                                  # greedy, so results are repeatable
        "stream": True,
        "stream_options": {"include_usage": True}
    }         # last chunk reports token counts
    t0 = time.perf_counter()
    
    first_token_at, text, completion_tokens, chunks = None, [], None, 0
    
    async with client.stream("POST", url + "/v1/chat/completions", json=body) as r:
        r.raise_for_status()
        async for line in r.aiter_lines():
            if not line.startswith("data: "):
                continue
            data = line[len("data: "):]
            if data == "[DONE]":
                break
            
            msg = json.loads(data)
            if msg.get("usage"):
                completion_tokens = msg["usage"]["completion_tokens"]
                
            for choice in msg.get("choices", []):
                piece = choice.get("delta", {}).get("content")
                if piece:
                    if first_token_at is None:
                        first_token_at = time.perf_counter()
                    text.append(piece)
                    chunks += 1
    
    end = time.perf_counter()
    n = completion_tokens if completion_tokens is not None else chunks
    ttft = (first_token_at or end) - t0
    itl = (end - first_token_at) / (n - 1) if first_token_at and n > 1 else None
    return {"ttft_s": ttft, "e2e_s": end - t0, "tokens": n, "itl_s": itl, "text": "".join(text)}

async def single_test():
    async with httpx.AsyncClient(timeout=300) as client:
        await stream_chat(client, url, "Say hi.")              # warm-up
        return await stream_chat(client, url, "Explain what a KV cache is in two sentences.")

r = await single_test()
print(f"TTFT {r['ttft_s']*1000:.0f} ms | inter-token {r['itl_s']*1000:.1f} ms | "
      f"{r['tokens']} tokens in {r['e2e_s']:.2f}s")
print("answer:", r["text"])
        

TTFT 30 ms | inter-token 6.5 ms | 35 tokens in 0.25s
answer: A KVS (Key-Value Store) is a type of caching system that stores data using keys and values, allowing for fast retrieval of data based on the key.


In [23]:
def fetch_metrics(url):
    """Read /metrics (Prometheus text) and add up each metric over all its labels."""
    out = {}
    for line in httpx.get(url + "/metrics", timeout=5).text.splitlines():
        if not line.startswith("vllm:"):
            continue
        name_and_labels, value = line.rsplit(" ", 1)
        name = name_and_labels.split("{")[0]
        try:
            out[name] = out.get(name, 0.0) + float(value)
        except ValueError:
            pass
    return out


def metric(m, base):
    """Counters may appear with or without a '_total' suffix; accept both."""
    return m.get(base, m.get(base + "_total", 0.0))


m = fetch_metrics(url)
for name in ["vllm:num_requests_running", "vllm:num_requests_waiting", "vllm:kv_cache_usage_perc"]:
    print(f"{name:32s} {metric(m, name)}")
for name in ["vllm:prompt_tokens", "vllm:generation_tokens"]:
    print(f"{name:32s} {metric(m, name):.0f}")
print(f"{'vllm:time_to_first_token (count)':32s} {metric(m, 'vllm:time_to_first_token_seconds_count'):.0f}")

vllm:num_requests_running        0.0
vllm:num_requests_waiting        0.0
vllm:kv_cache_usage_perc         0.0
vllm:prompt_tokens               936
vllm:generation_tokens           585
vllm:time_to_first_token (count) 26


In [26]:
async def load_test(url, concurrency, n_requests):
    todo, results, samples = list(range(n_requests)), [], []
    stop = asyncio.Event()

    async def poll_metrics():                       # sample the server state while the test runs
        while not stop.is_set():
            try:
                samples.append(await asyncio.to_thread(fetch_metrics, url))
            except Exception:
                pass
            await asyncio.sleep(0.5)

    async def user(client):
        while todo:
            q = QUESTIONS[todo.pop(0) % len(QUESTIONS)]
            results.append(await stream_chat(client, url, q))

    async with httpx.AsyncClient(timeout=600) as client:
        poller = asyncio.create_task(poll_metrics())
        t0 = time.perf_counter()
        await asyncio.gather(*[user(client) for _ in range(concurrency)])
        wall = time.perf_counter() - t0
        stop.set()
        await poller

    def pct(values, q):
        v = sorted(values)
        return v[min(len(v) - 1, int(q * len(v)))]

    ttft = [x["ttft_s"] for x in results]
    e2e = [x["e2e_s"] for x in results]
    itl = [x["itl_s"] for x in results if x["itl_s"] is not None]
    return {"concurrency": concurrency, "requests": len(results),
            "tok_per_s": sum(x["tokens"] for x in results) / wall,
            "req_per_s": len(results) / wall,
            "ttft_p50": pct(ttft, 0.5), "ttft_p95": pct(ttft, 0.95),
            "e2e_p50": pct(e2e, 0.5), "e2e_p95": pct(e2e, 0.95),
            "itl_mean": statistics.mean(itl) if itl else float("nan"),
            "max_running": max((metric(s, "vllm:num_requests_running") for s in samples), default=0),
            "max_waiting": max((metric(s, "vllm:num_requests_waiting") for s in samples), default=0),
            "max_kv_usage": max((metric(s, "vllm:kv_cache_usage_perc") for s in samples), default=0)}


def print_row(r, label=""):
    print(f"{label}conc {r['concurrency']:>3} | {r['tok_per_s']:7.1f} tok/s | {r['req_per_s']:5.2f} req/s | "
          f"TTFT p50 {r['ttft_p50']*1000:6.0f}ms p95 {r['ttft_p95']*1000:6.0f}ms | "
          f"E2E p50 {r['e2e_p50']:5.2f}s p95 {r['e2e_p95']:5.2f}s | ITL {r['itl_mean']*1000:5.1f}ms | "
          f"running≤{r['max_running']:.0f} waiting≤{r['max_waiting']:.0f} KV≤{r['max_kv_usage']:.1%}")


vllm_results = []
for c in [1, 4, 8, 16, 32]:
    r = await load_test(url, c, n_requests=max(16, 2 * c))
    vllm_results.append(r)
    print_row(r)

conc   1 |   154.8 tok/s |  5.11 req/s | TTFT p50     26ms p95     43ms | E2E p50  0.29s p95  0.33s | ITL   5.6ms | running≤1 waiting≤0 KV≤0.0%
conc   4 |   468.0 tok/s | 15.44 req/s | TTFT p50     44ms p95     80ms | E2E p50  0.30s p95  0.35s | ITL   5.8ms | running≤3 waiting≤0 KV≤0.0%
conc   8 |   674.6 tok/s | 22.26 req/s | TTFT p50     85ms p95     93ms | E2E p50  0.30s p95  0.42s | ITL   6.4ms | running≤2 waiting≤0 KV≤0.0%
conc  16 |  1104.1 tok/s | 36.42 req/s | TTFT p50    105ms p95    114ms | E2E p50  0.37s p95  0.52s | ITL   7.9ms | running≤5 waiting≤0 KV≤0.0%
conc  32 |  1139.9 tok/s | 37.61 req/s | TTFT p50    223ms p95    322ms | E2E p50  0.68s p95  1.06s | ITL  16.1ms | running≤28 waiting≤0 KV≤0.1%


In [27]:
before = fetch_metrics(url)
r = await load_test(url, 16, n_requests=32)
after = fetch_metrics(url)

def delta(base):
    return metric(after, base) - metric(before, base)

def mean_from_histogram(base):
    count = delta(base + "_count")
    return delta(base + "_sum") / count if count else float("nan")

print("SERVER-SIDE view of the same 32 requests (from /metrics):")
print(f"  mean queue time   : {mean_from_histogram('vllm:request_queue_time_seconds') * 1000:7.1f} ms")
print(f"  mean TTFT         : {mean_from_histogram('vllm:time_to_first_token_seconds') * 1000:7.1f} ms")
print(f"  mean E2E latency  : {mean_from_histogram('vllm:e2e_request_latency_seconds'):7.2f} s")
print(f"  prompt tokens     : {delta('vllm:prompt_tokens'):7.0f}")
print(f"  generated tokens  : {delta('vllm:generation_tokens'):7.0f}")
queries = delta("vllm:prefix_cache_queries")
print(f"  prefix cache hits : {delta('vllm:prefix_cache_hits'):.0f} / {queries:.0f} tokens"
      + (f" ({delta('vllm:prefix_cache_hits') / queries:.0%})" if queries else ""))
print(f"  preemptions       : {delta('vllm:num_preemptions'):.0f}")
print("\nCLIENT-SIDE view:")
print_row(r)

SERVER-SIDE view of the same 32 requests (from /metrics):
  mean queue time   :     0.0 ms
  mean TTFT         :    60.9 ms
  mean E2E latency  :    0.32 s
  prompt tokens     :    1238
  generated tokens  :     970
  prefix cache hits : 992 / 1238 tokens (80%)
  preemptions       : 0

CLIENT-SIDE view:
conc  16 |  1036.9 tok/s | 34.21 req/s | TTFT p50    111ms p95    119ms | E2E p50  0.36s p95  0.58s | ITL   8.5ms | running≤8 waiting≤0 KV≤0.1%


In [28]:
stop_vllm(proc)
proc, url = start_vllm(extra_args=["--max-num-seqs", "4"], port=8001)
try:
    limited = await load_test(url, 16, n_requests=32)
    print_row(limited, "max-num-seqs=4 | ")
    normal = next(x for x in vllm_results if x["concurrency"] == 16)
    print_row(normal, "default        | ")
finally:
    stop_vllm(proc)

vLLM stopped
starting: vllm serve Qwen/Qwen2.5-0.5B-Instruct --port 8001 --dtype half --max-model-len 2048 --gpu-memory-utilization 0.85 --max-num-seqs 4
vLLM ready after 106s
  log: To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.8528. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
  log: (EngineCore pid=15247) INFO 10-12 13:00:40 [kv_cache_utils.py:2464] GPU KV cache size: 964,880 tokens, Maximum concurrency for 2,048 tokens per request: 471.13x
max-num-seqs=4 | conc  16 |   455.6 tok/s | 15.03 req/s | TTFT p50    719ms p95    883ms | E2E p50  0.86s p95  1.24s | ITL   6.7ms | running≤3 waiting≤12 KV≤0.0%
default        | conc  16 |  1104.1 tok/s | 36.42 req/s | TTFT p50    105ms p95    114ms | E2E p50  0.37s p95  0.52s | ITL   7.9ms | running≤5 waiting≤0 KV≤0.0%
vLLM stopped


In [29]:
%%writefile offline_vllm.py
import time
from vllm import LLM, SamplingParams

QUESTIONS = [
    "Answer with one word: what colour is the sky on a clear day?",
    "What is 12 times 7? Reply with only the number.",
    "Write three sentences about why the ocean is salty.",
    "List five fruits, comma separated.",
    "Explain in detail how a refrigerator keeps food cold.",
    "Say hello.",
    "Give a short definition of photosynthesis.",
    "Write a short paragraph about the history of the bicycle.",
    "What is the capital of Japan? One word.",
    "Describe how rain forms, step by step.",
    "Translate 'good morning' into French.",
    "Write a haiku about coffee.",
    "Explain what a database index is and why it speeds up queries.",
    "Name the planet closest to the sun.",
    "Give three tips for writing clean Python code.",
    "Summarise the water cycle in two sentences.",
]

if __name__ == "__main__":
    llm = LLM(model="Qwen/Qwen2.5-0.5B-Instruct", dtype="half", max_model_len=2048,
              gpu_memory_utilization=0.85)
    params = SamplingParams(temperature=0, max_tokens=48)
    conversations = [[{"role": "user", "content": q}] for q in QUESTIONS]

    llm.chat(conversations[:2], params, use_tqdm=False)                 # warm-up
    t0 = time.perf_counter()
    outputs = llm.chat(conversations, params, use_tqdm=False)
    secs = time.perf_counter() - t0

    tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    print(f"OFFLINE vLLM: {len(outputs)} requests, {tokens} tokens in {secs:.2f}s -> {tokens / secs:.1f} tok/s")
    for q, o in list(zip(QUESTIONS, outputs))[:3]:
        print(f"- {q}\n  -> {o.outputs[0].text.strip()[:100]!r}")

Writing offline_vllm.py


In [30]:
!python offline_vllm.py 2>&1 | grep -E "^(OFFLINE|- |  ->|Traceback|.*Error)"

OFFLINE vLLM: 16 requests, 453 tokens in 0.40s -> 1124.0 tok/s
- Answer with one word: what colour is the sky on a clear day?
  -> 'Blue'
- What is 12 times 7? Reply with only the number.
  -> '84'
- Write three sentences about why the ocean is salty.
  -> 'The ocean is salty because it is composed of water, which is a saltwater solution. The saltwater sol'
